# A2 · One Row per Person

*Path A: Write It Yourself, notebook 2 of 6 · about 20 to 30 minutes*

**In this notebook you will learn to:**

- Reduce a table with many records per person to one row per person
- Do a calculation separately for each person with `group_by()`
- Merge three tables into one and check the result

**Before this notebook:** [A1: Cleaning Real Data](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/A1_Cleaning_Real_Data.ipynb).

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. Where we are

The goal is one table with one row per person. `person_df` is already like that. But at the end of the Foundations you saw that `condition_df` and `drug_df` have several records per person, and that joining them directly multiplies the rows.

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal. It also repeats the recoding you did in notebook A1, so that you start from clean data.

In [ ]:
library(tidyverse)

person_df    <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/demographic_data.csv")
condition_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/conditions.csv")
drug_df      <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/drug_exposures.csv")

# Catch-up: the recoding from notebook A1
person_df <- person_df %>%
  mutate(
    race = case_when(
      race == "Black or African American" ~ "Black",
      race == "Skip" ~ "Unknown",
      race %in% c("More than one population", "None of these") ~ "Other",
      .default = race
    ),
    sex_at_birth = case_when(
      sex_at_birth %in% c("Intersex", "Prefer not to answer", "Skip") ~ "Other",
      .default = sex_at_birth
    )
  )

## 2. Look at one person

Before writing code for 100 people, look closely at one. Here are all the diagnosis records for person 8.

In [ ]:
condition_df %>% filter(person_id == 8)

Four records, on four dates. For this study, the date that matters is when the condition **first** appeared: the earliest one.

## 3. Keeping the earliest row: `slice_min()`

`slice_min()` keeps the row with the smallest value in a column you choose. For a date column, the smallest value is the earliest date. `n = 1` asks for one row.

In [ ]:
condition_df %>%
  filter(person_id == 8) %>%
  slice_min(condition_start_date, n = 1)

One row is left: the earliest record, dated 2025-06-09. (The data are simulated, so do not read anything into the dates.)

## 4. Doing it for every person: `group_by()`

Without the `filter()`, `slice_min()` would return the single earliest record in the **whole table**, which is not what you want. You want the earliest record **for each person**.

`group_by(person_id)` tells R: "from now on, do each step separately for each person". After it, `slice_min()` picks the earliest row within each person's records.

In [ ]:
earliest_condition <- condition_df %>%
  group_by(person_id) %>%                          # for each person...
  slice_min(condition_start_date, n = 1, with_ties = FALSE) %>%   # ...keep the earliest record
  ungroup()                                        # stop working person by person

Two details:

- **`with_ties = FALSE`**: if a person has two records on the same earliest date, keep only one of them. Without this, both would be kept and that person would still have two rows.
- **`ungroup()`**: switches the "for each person" behavior off again. Always finish a grouped step with it, so later steps are not affected.

### Check

In [ ]:
nrow(earliest_condition)                    # rows
n_distinct(earliest_condition$person_id)    # different people

In [ ]:
earliest_condition %>% filter(person_id == 8)   # the person you looked at by hand

30 rows for 30 people, and person 8 has the date you found by hand. Checking a result against a case you worked out yourself is one of the most reliable tests there is.

**Your turn 1.** Do the same for the prescription records. Create `earliest_drug` from `drug_df`, keeping each person's earliest `drug_exposure_start_date`. Then check the number of rows against the number of different people.

*Expected result: 35 rows, 35 different people*

In [ ]:
# Your code here

## 5. Merge the three tables

Now every table has one row per person, so the joins are safe. Start from `person_df` and add the other two with `left_join()`, so that all 100 people are kept.

In [ ]:
combined_df <- person_df %>%
  left_join(earliest_condition, by = "person_id") %>%
  left_join(earliest_drug, by = "person_id")

(If this cell gives the error `object 'earliest_drug' not found`, finish the exercise above first.)

### Check

In [ ]:
nrow(combined_df)                    # should still be 100
n_distinct(combined_df$person_id)    # should also be 100

In [ ]:
head(combined_df)

In [ ]:
colSums(is.na(combined_df))

100 rows, 100 people. The missing values are expected and meaningful here: 70 people have no diagnosis record, and 65 have no prescription record. In this table, `NA` in a date column means "this never happened", as far as the records show.

`!` means "not". So `!is.na(x)` means "x is **not** missing".

In [ ]:
combined_df %>%
  filter(!is.na(condition_start_date)) %>%   # people who do have a diagnosis date
  nrow()

**Your turn 2.** How many people have **both** a diagnosis date and a prescription date? Use `filter()` with `!is.na()` twice, joined by `&`.

*Expected result: 21*

In [ ]:
# Your code here

## 6. Optional extra: survey answers, from long to wide

This section is useful if your project uses *All of Us* survey data. You can skip it and come back later.

Survey data arrives in a **long** layout: one row per person per question.

In [ ]:
survey_long <- data.frame(
  person_id = c(1, 1, 1, 2, 2, 2),
  question  = c("smoked_100_cigs", "alcohol_frequency", "general_health",
                "smoked_100_cigs", "alcohol_frequency", "general_health"),
  answer    = c("Yes", "Never", "Good", "No", "Monthly", "Fair")
)
survey_long

For analysis you want a **wide** layout: one row per person, one column per question. `pivot_wider()` does this. You tell it which column holds the new column names and which holds the values.

In [ ]:
survey_wide <- survey_long %>%
  pivot_wider(
    id_cols     = person_id,   # one row for each of these
    names_from  = question,    # new column names come from here
    values_from = answer       # the values to fill in come from here
  )
survey_wide

---
## Recap

| Code | What it does |
|---|---|
| `slice_min(date, n = 1, with_ties = FALSE)` | Keep the one row with the earliest date |
| `group_by(person_id)` | Do the following steps separately for each person |
| `ungroup()` | Stop working group by group |
| `a %>% left_join(b, by = "person_id")` | Add columns from b, keeping every row of a |
| `!is.na(x)` | x is not missing |
| `pivot_wider(...)` | Reshape long data to one row per person |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Your turn 1**

```r
earliest_drug <- drug_df %>%
  group_by(person_id) %>%
  slice_min(drug_exposure_start_date, n = 1, with_ties = FALSE) %>%
  ungroup()

nrow(earliest_drug)
n_distinct(earliest_drug$person_id)
```

**Your turn 2**

```r
combined_df %>%
  filter(!is.na(condition_start_date) & !is.na(drug_exposure_start_date)) %>%
  nrow()
```

---
## Next

Continue with [A3: Dates and New Variables](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/A3_Dates_and_New_Variables.ipynb).